<a href="https://colab.research.google.com/github/dataprogpy/code-samples/blob/dev/starter_files/07_robust_modeling_workflow.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Robust Modeling Workflow

In this lesson we extend the six-step workflow to introduce critical modeling best practices. In this session, we will move from classfication to regression.We will reuse the King County Housing dataset introduced in the last module for this purpose.

In [23]:
# tools from standard library
import functools
from pathlib import Path

# tools for data wrangling
import numpy as np
import polars as pl
import polars.selectors as cs

import geopandas as gpd

# tools for plotting
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

# crossvalidation
from sklearn.model_selection import (
    train_test_split,
    GroupKFold,
    GroupShuffleSplit,
    KFold,
    ShuffleSplit,
    StratifiedGroupKFold,
    StratifiedKFold,
    StratifiedShuffleSplit,
    TimeSeriesSplit,
)

# ingredients for building a pipeline
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    StandardScaler,
    OneHotEncoder,
    TargetEncoder,
    FunctionTransformer,
)
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer

# base type for building custom transformer
from sklearn.base import BaseEstimator, TransformerMixin

# models
from sklearn.dummy import DummyRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.linear_model import LinearRegression

# metrics
from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_error


# configure scikit-learn to output polars dataframe
from sklearn import set_config
set_config(transform_output='polars')

## Understanding cross-validation

In [58]:
data_root = Path("/content/drive/MyDrive/dataprogpy/data")
house_data_path = Path("kc_house_data.csv")
county_file_path = Path("kingcounty/King_county_zip.shp")
schdst_file_path = Path("School_Districts_in_King_County___schdst_area/School_Districts_in_King_County___schdst_area.shp")

In [59]:
kc_zip = gpd.read_file(data_root / county_file_path) # zipcode shapes
kc_schdst = gpd.read_file(data_root / schdst_file_path) # school district shapes

In [40]:
kc_zip_schdst = pl.from_pandas(
    kc_zip.sjoin(
        kc_schdst.to_crs("EPSG:4326"),
        how="left",
        predicate="intersects")[["ZIP","NAME"]]
    ).select(
        pl.col('ZIP').cast(pl.String).alias("zipcode"),
        pl.col("NAME").cast(pl.String).alias("school_district")
    )

kc_zip_schdst.select(
    pl.col("zipcode").n_unique(),
    pl.col("school_district").n_unique()
)

zipcode,school_district
u32,u32
85,20


In [60]:
# Load the dataset
raw = pl.read_csv(data_root / house_data_path)
raw.head()

id,date,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,grade,sqft_above,sqft_basement,yr_built,yr_renovated,zipcode,lat,long,sqft_living15,sqft_lot15
i64,str,f64,i64,f64,i64,i64,f64,i64,i64,i64,i64,i64,i64,i64,i64,i64,f64,f64,i64,i64
7129300520,"""20141013T000000""",221900.0,3,1.0,1180,5650,1.0,0,0,3,7,1180,0,1955,0,98178,47.5112,-122.257,1340,5650
6414100192,"""20141209T000000""",538000.0,3,2.25,2570,7242,2.0,0,0,3,7,2170,400,1951,1991,98125,47.721,-122.319,1690,7639
5631500400,"""20150225T000000""",180000.0,2,1.0,770,10000,1.0,0,0,3,6,770,0,1933,0,98028,47.7379,-122.233,2720,8062
2487200875,"""20141209T000000""",604000.0,4,3.0,1960,5000,1.0,0,0,5,7,1050,910,1965,0,98136,47.5208,-122.393,1360,5000
1954400510,"""20150218T000000""",510000.0,3,2.0,1680,8080,1.0,0,0,3,8,1680,0,1987,0,98074,47.6168,-122.045,1800,7503


In [61]:
kc_schdst.columns

Index(['OBJECTID', 'SCHDST', 'NAME', 'DSTNUM', 'Shape_Leng', 'Shape_Area',
       'geometry'],
      dtype='object')

In [63]:
raw_pd = raw.to_pandas()
raw_pd = gpd.GeoDataFrame(
        raw_pd,
        geometry=gpd.points_from_xy(raw_pd.long, raw_pd.lat),
        crs="EPSG:4326"
        ).sjoin(
            kc_schdst.to_crs("EPSG:4326"),
            how="left",
            predicate="intersects")
raw_pd.head()

,id,date,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,...,sqft_living15,sqft_lot15,geometry,index_right,OBJECTID,SCHDST,NAME,DSTNUM,Shape_Leng,Shape_Area
0,7129300520,20141013T000000,221900.0,3,1.00,1180,5650,1.0,0,0,...,1340,5650,POINT (-122.257 47.5112),0.0,1.0,1,Seattle,17001,435832.057323,2.570871e+09
1,6414100192,20141209T000000,538000.0,3,2.25,2570,7242,2.0,0,0,...,1690,7639,POINT (-122.319 47.721),0.0,1.0,1,Seattle,17001,435832.057323,2.570871e+09
2,5631500400,20150225T000000,180000.0,2,1.00,770,10000,1.0,0,0,...,2720,8062,POINT (-122.233 47.7379),18.0,19.0,417,Northshore,17417,188908.404351,1.062970e+09
3,2487200875,20141209T000000,604000.0,4,3.00,1960,5000,1.0,0,0,...,1360,5000,POINT (-122.393 47.5208),0.0,1.0,1,Seattle,17001,435832.057323,2.570871e+09
4,1954400510,20150218T000000,510000.0,3,2.00,1680,8080,1.0,0,0,...,1800,7503,POINT (-122.045 47.6168),16.0,17.0,414,Lake Washington,17414,309878.307056,1.918889e+09


In [70]:
raw = pl.from_pandas(raw_pd[raw.columns + ["NAME"]]).select(
    pl.col(raw.columns),
    pl.col("NAME").alias("school_district")
)

In [71]:
raw.shape

(21613, 22)

In [72]:
def tweak_housing(df):
    return (df
            .with_columns(
                zipcode=pl.col('zipcode').cast(pl.String).cast(pl.Categorical),
                school_district=pl.col('school_district').cast(pl.Categorical),
                date= pl.col('date').str.strptime(pl.Date, format="%Y%m%dT%H%M%S").alias('sale_date'),
                yr_renovated=pl.col('yr_renovated').replace(0, None),
            )
            .select(
                pl.col(
                    'id', 'price', 'bedrooms', 'bathrooms', 'sqft_living', 'sqft_lot', 'floors', 'waterfront', 'view', 'condition', 'grade', 'sqft_above', 'sqft_basement', 'yr_built', 'yr_renovated', 'zipcode', 'lat', 'long', 'sqft_living15', 'sqft_lot15', 'date', 'school_district'
                     )
                )
            )
numeric_features = ['bedrooms', 'bathrooms', 'sqft_living']
categorical_features = ['zipcode', 'school_district']

In [73]:
tweak_housing(raw).select(pl.col(numeric_features)).head()

bedrooms,bathrooms,sqft_living
i64,f64,i64
3,1.0,1180
3,2.25,2570
2,1.0,770
4,3.0,1960
3,2.0,1680


In [74]:
tweak_housing(raw).select(pl.col(categorical_features)).head()

zipcode,school_district
cat,cat
"""98178""","""Seattle"""
"""98125""","""Seattle"""
"""98028""","""Northshore"""
"""98136""","""Seattle"""
"""98074""","""Lake Washington"""


In [20]:
std = StandardScaler()
std.fit_transform(tweak_housing(raw).select(numeric_features)).head()

bedrooms,bathrooms,sqft_living
f64,f64,f64
-0.398737,-1.447464,-0.979835
-0.398737,0.175607,0.533634
-1.473959,-1.447464,-1.426254
0.676485,1.149449,-0.13055
-0.398737,-0.149007,-0.435422


In [21]:
num_pipeline = Pipeline([
     ('std', StandardScaler())])

num_pipeline.fit_transform(
    tweak_housing(raw)
    .select(numeric_features)
).head()

bedrooms,bathrooms,sqft_living
f64,f64,f64
-0.398737,-1.447464,-0.979835
-0.398737,0.175607,0.533634
-1.473959,-1.447464,-1.426254
0.676485,1.149449,-0.13055
-0.398737,-0.149007,-0.435422


In [22]:
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('std', StandardScaler())])

num_pipeline.fit_transform(
    tweak_housing(raw)
    .select(numeric_features)
).head()

bedrooms,bathrooms,sqft_living
f64,f64,f64
-0.398737,-1.447464,-0.979835
-0.398737,0.175607,0.533634
-1.473959,-1.447464,-1.426254
0.676485,1.149449,-0.13055
-0.398737,-0.149007,-0.435422


In [76]:
categorical_features = ['zipcode', 'school_district']

In [80]:
cat_features = ['zipcode']

ohe = OneHotEncoder(handle_unknown='ignore',
                    sparse_output=False, max_categories=10)

ohe.fit_transform(
    tweak_housing(raw)
    .select(cat_features)
).head()

zipcode_98023,zipcode_98034,zipcode_98038,zipcode_98042,zipcode_98052,zipcode_98103,zipcode_98115,zipcode_98117,zipcode_98118,zipcode_infrequent_sklearn
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


In [82]:
cat_features = ['zipcode']

cat_pipeline = Pipeline(steps=[
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False))])

cat_pipeline.set_params(cat__max_categories=10)
cat_pipeline.fit_transform(
    tweak_housing(raw)
    .select(cat_features)
).head()

zipcode_98023,zipcode_98034,zipcode_98038,zipcode_98042,zipcode_98052,zipcode_98103,zipcode_98115,zipcode_98117,zipcode_98118,zipcode_infrequent_sklearn
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


In [86]:
cat_pipeline = Pipeline([
    ('target', TargetEncoder()),
    # ('std', StandardScaler()),
    ])

cat_pipeline.fit_transform(
    tweak_housing(raw).select('zipcode'), tweak_housing(raw).select('price')
)

/usr/local/lib/python3.11/dist-packages/sklearn/model_selection/_split.py:805: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(


zipcode_75000.0,zipcode_78000.0,zipcode_80000.0,zipcode_81000.0,zipcode_82000.0,zipcode_82500.0,zipcode_83000.0,zipcode_84000.0,zipcode_85000.0,zipcode_86500.0,zipcode_89000.0,zipcode_89950.0,zipcode_90000.0,zipcode_92000.0,zipcode_95000.0,zipcode_96500.0,zipcode_99000.0,zipcode_100000.0,zipcode_102500.0,zipcode_104950.0,zipcode_105000.0,zipcode_105500.0,zipcode_106000.0,zipcode_107000.0,zipcode_109000.0,zipcode_109500.0,zipcode_110000.0,zipcode_110700.0,zipcode_111300.0,zipcode_112000.0,zipcode_114000.0,zipcode_114975.0,zipcode_115000.0,zipcode_118000.0,zipcode_118125.0,zipcode_119500.0,zipcode_119900.0,…,zipcode_2983000.0,zipcode_2998000.0,zipcode_3000000.0,zipcode_3065000.0,zipcode_3070000.0,zipcode_3075000.0,zipcode_3100000.0,zipcode_3120000.0,zipcode_3168750.0,zipcode_3200000.0,zipcode_3204000.0,zipcode_3278000.0,zipcode_3300000.0,zipcode_3345000.0,zipcode_3395000.0,zipcode_3400000.0,zipcode_3418800.0,zipcode_3567000.0,zipcode_3600000.0,zipcode_3635000.0,zipcode_3640900.0,zipcode_3650000.0,zipcode_3710000.0,zipcode_3800000.0,zipcode_3850000.0,zipcode_4000000.0,zipcode_4208000.0,zipcode_4489000.0,zipcode_4500000.0,zipcode_4668000.0,zipcode_5110800.0,zipcode_5300000.0,zipcode_5350000.0,zipcode_5570000.0,zipcode_6885000.0,zipcode_7062500.0,zipcode_7700000.0
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.004242,0.0,0.0,0.0,0.0,0.004314,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00339,…,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,…,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,…,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,…,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,…,0.0,0.0,0.0,0.0,0.0,0.002539,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,…,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
0.0,0.0,0.0,0.0,0.003243,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.003243,0.0,0.0,0.003243,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.003243,0.01075,0.0,0.0,0.0,0.0,…,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0

## References

1. [Comparing cross-validation strategies](https://scikit-learn.org/stable/auto_examples/model_selection/plot_cv_indices.html)
1.